|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Continuous batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: write the iteration-level scheduler<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(1)

Write the scheduler.

You need one function, one loop, and a hook for the [admission](../../GLOSSARY.md#admission) policy. The hook
lets you change the policy in Exercise 4.

This is stage 05 of the ladder, with a counter in place of the GPU. Make a
scheduler correct first. Make it fast later.

In [ ]:
### run this cell

NUM_REQUESTS = 3000
NUM_SLOTS = 64
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=NUM_REQUESTS).astype(int) + 1
capacity = NUM_SLOTS / lengths.mean()
arrivals = np.cumsum(rng.exponential(1/(0.7*capacity), size=NUM_REQUESTS))

print(f'{NUM_REQUESTS} requests, mean {lengths.mean():.0f} tokens, {NUM_SLOTS} slots')

# Exercise 1: the step loop

Four things happen every step, in this order:

1. arrivals join the waiting queue
2. free slots take requests from that queue
3. every running sequence makes one token
4. a sequence that just finished leaves its slot at once

Step 4 is the whole idea. A static batcher does step 4 at the end of the
batch.

In [ ]:
def run(arrivals, lengths, NUM_SLOTS, pick):
  """pick(waiting, lengths) -> index INTO `waiting` of the one to admit."""
  finish_steps, start_steps = np.zeros(len(lengths)), np.zeros(len(lengths))
  step, next_request, running, waiting = 0.0, 0, {}, []
  busy = []

  while next_request < len(lengths) or waiting or running:

    # 1. everything that has arrived by now joins the waiting queue
    while next_request < len(lengths) and arrivals[next_request] <= step:
      

    # 2. fill every free slot from the waiting queue, using `pick`
    while len(running) < NUM_SLOTS and waiting:
      request = 
      

    if not running:
      step = arrivals[next_request]
      continue  # nothing to do, skip ahead

    busy.append(len(running))

    # 3. ONE decode step. Every running sequence emits one token.
    step += 1.0
    for request in list(running):
      
      # 4. anything that just finished leaves its slot NOW,
      #    not at the end of some batch
      

  return finish_steps, start_steps, np.array(busy)

fcfs = lambda waiting, lengths: 0     # admit the oldest request
finish_steps, start_steps, busy = run(arrivals, lengths, NUM_SLOTS, fcfs)
print(f'finished at step {finish_steps.max():,.0f}')

# Exercise 2: prove that it does not lie

A scheduler can drop requests or hand out extra tokens and still report a
plausible throughput number. Check the invariants.

In [ ]:
# every request finished, got exactly its own number of tokens, never
# started before it arrived, and the slot limit was never exceeded
assert (finish_steps > 0).all(), 'some request never finished'
assert , 'wrong number of tokens somewhere'
assert , 'a request started before it arrived'
assert , 'more sequences running than there are slots'
print('all checks passed')
print(f'mean occupancy {busy.mean():.1f} of {NUM_SLOTS} slots ({100*busy.mean()/NUM_SLOTS:.0f}%)')

# Exercise 3: against static batching

In [ ]:
def static_batching(arrivals, lengths, NUM_SLOTS):
  finish = np.zeros(len(lengths))
  step = 0.0
  first_waiting = 0
  while first_waiting < len(lengths):
    batch = np.arange(first_waiting, min(first_waiting+NUM_SLOTS, len(lengths)))
    step = max(step, arrivals[batch[-1]])
    longest = lengths[batch].max()
    finish[batch] = step + longest
    step += longest
    first_waiting += NUM_SLOTS
  return finish

static_finish = static_batching(arrivals, lengths, NUM_SLOTS)
static_latency, continuous_latency = , 

print(f"{'':<12} {'makespan':>10} {'p50 lat':>9} {'p99 lat':>9}")
print(f"{'static':<12} {static_finish.max():>10,.0f} {np.median(static_latency):>9,.0f} {np.percentile(static_latency,99):>9,.0f}")
print(f"{'continuous':<12} {finish_steps.max():>10,.0f} {np.median(continuous_latency):>9,.0f} {np.percentile(continuous_latency,99):>9,.0f}")
print(f'\nthroughput {static_finish.max()/finish_steps.max():.2f}x, p99 latency {np.percentile(static_latency,99)/np.percentile(continuous_latency,99):.0f}x better')

# Exercise 4: change the admission policy

First-come-first-served is one choice. Now admit the shortest request instead.
Look at each part of the distribution, not at the average.

Do one thing first. At 70 percent load your waiting queue is empty almost
every step, so `pick` never has a choice and every policy scores the same.
Load the server to 120 percent, and the question becomes real. A scheduler is
a scheduler only when there is something to schedule.

In [ ]:
# admit the request with the fewest tokens left to generate.
# `pick` receives the waiting list and the lengths array.
sjf = lambda waiting, lengths: 

sjf_finish, _, _ = run(busy_arrive, lengths, NUM_SLOTS, sjf)

print(f"{'policy':<22} {'makespan':>10} {'p50':>8} {'p99':>9} {'worst':>9}")
for policy_name, policy_finish in (('first come first served', fcfs_finish), ('shortest job first', sjf_finish)):
  latency = policy_finish - busy_arrive
  print(f'{policy_name:<22} {policy_finish.max():>10,.0f} {np.median(latency):>8,.0f} {np.percentile(latency,99):>9,.0f} {latency.max():>9,.0f}')

### Before you open the solution

1. Continuous batching beat static batching on throughput **and** on latency.
   These two usually trade against each other. Why did they not trade here?
2. Compare the `p50` and `worst` columns between the two policies. Who gains
   under shortest-job-first? Who pays?
3. Look at the value that your `sjf` function reads. Can a real server know
   that value? What must it do instead?